### Desarrollo Challenge 4 - Webscrapping

#### Imports y Configuraciones

In [ ]:
import os
import re
import requests
import psycopg2

from dotenv import load_dotenv
from bs4 import BeautifulSoup

#### Scrapeo de Categorias

In [ ]:
url = "https://books.toscrape.com/"

response = requests.get(url)
soup = BeautifulSoup(response.text, "html.parser")

categorias = soup.find("div", class_="side_categories").find_all("a")

for categoria in categorias:
    nombre = categoria.text.strip()

    if nombre != "Books":
        print(nombre)

#### Scrapeo de Libros

In [ ]:
cantidad_de_libros = 0

for pagina in range(1, 51):
    url = f'https://books.toscrape.com/catalogue/page-{pagina}.html'

    response = requests.get(url)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, 'html.parser')

    libros = soup.find_all('article', class_='product_pod')

    for libro in libros:
        cantidad_de_libros += 1
        
        titulo = libro.h3.a['title']
        precio = libro.find('p', class_='price_color').text
        rating_texto = libro.p['class'][1]

print(f'Cantidad de libros scrapeados: {cantidad_de_libros}')

#### Conexion a la Base de Datos

In [ ]:
load_dotenv(override=True)

conn = psycopg2.connect(
    dbname = os.getenv('DB_NAME'),
    user = os.getenv('DB_USER'),
    password = os.getenv('DB_PASSWORD'),
    host = os.getenv('DB_HOST'),
    port = os.getenv('DB_PORT')
)

cursor = conn.cursor()

print("Conectado a PostgreSQL")

#### Cargar Schema a la Base de Datos

In [ ]:
with open('schema.sql', 'r') as schema:
    ddl = schema.read()

cursor.execute(ddl)
conn.commit()

print("Schema creado")